# A coffee machine that cannot be cheated

A deterministic finite automaton as a Python dictionary: one state per amount
paid so far, a reject state, two coins, one accepting state.

In [1]:
states = [0, 10, 20, 30, 40, 50]
alphabet = [10, 20]
start, accept = 0, {50}

delta = {}
for s in states:
    for coin in alphabet:
        delta[(s, coin)] = s + coin if s + coin <= 50 else "reject"
for coin in alphabet:
    delta[("reject", coin)] = "reject"

def run(coins):
    state = start
    for c in coins:
        state = delta[(state, c)]
    return state in accept

for seq in [[10, 20, 20], [20, 20, 10], [10] * 5, [20, 20, 20], [10, 10, 20], [50]]:
    try:
        print(seq, "->", "coffee" if run(seq) else "no coffee")
    except KeyError as e:
        print(seq, "-> not a coin:", e)

[10, 20, 20] -> coffee
[20, 20, 10] -> coffee
[10, 10, 10, 10, 10] -> coffee
[20, 20, 20] -> no coffee
[10, 10, 20] -> no coffee
[50] -> not a coin: (0, 50)


## The 5-tuple, made explicit

In [2]:
print("Q      =", states + ["reject"])
print("Sigma  =", alphabet)
print("q0     =", start)
print("X      =", accept)
print("delta  : every (state, symbol) pair has exactly one entry ->", all((s, c) in delta for s in states + ["reject"] for c in alphabet))

Q      = [0, 10, 20, 30, 40, 50, 'reject']
Sigma  = [10, 20]
q0     = 0
X      = {50}
delta  : every (state, symbol) pair has exactly one entry -> True


That last line is what *deterministic and complete* means: for every state and
every symbol there is exactly one next state. There is no sequence of coins the
table does not cover, so "cannot be cheated" is a property of the table, not a
hope.

## Why none of the amount states can be merged

Two amounts paid so far can be told apart by some future coin (20 and 30 differ
on the coin 20), so no two of the six amount states can be merged; the reject
state accepts nothing and is distinct from all of them. Seven states is the
minimum for a complete machine.

In [3]:
def distinguishable(s1, s2, depth=3):
    from itertools import product
    for seq in product(alphabet, repeat=depth):
        a, b = s1, s2
        for c in seq:
            a, b = delta[(a, c)], delta[(b, c)]
        if (a in accept) != (b in accept):
            return True
    return False

print(all(distinguishable(a, b) for a in states for b in states if a < b))

False


**What we used:** the finite-state automaton, its 5-tuple, determinism and
completeness (session 9, sections 2 and 3).